In [1]:
import os
import shutil

# ==========================
# PATHS
# ==========================
source_dataset = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\Mendeley Waste Classification"
new_dataset = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\smart_city_dataset"

# Create new folders
rec_path = os.path.join(new_dataset, "recyclable")
non_rec_path = os.path.join(new_dataset, "non_recyclable")

os.makedirs(rec_path, exist_ok=True)
os.makedirs(non_rec_path, exist_ok=True)

# ==========================
# CATEGORY MAPPING
# ==========================
recyclable_classes = [
    "recyclable"
]

non_recyclable_classes = [
    "organic"
]

# ==========================
# COPY FILES
# ==========================
def copy_images(classes, destination_folder):
    for cls in classes:
        class_path = os.path.join(source_dataset, cls)

        if os.path.exists(class_path):
            for img in os.listdir(class_path):
                src = os.path.join(class_path, img)

                if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                    dst = os.path.join(
                        destination_folder,
                        f"{cls}_{img}"
                    )

                    shutil.copy(src, dst)

copy_images(recyclable_classes, rec_path)
copy_images(non_recyclable_classes, non_rec_path)

print("Dataset converted successfully!")

Dataset converted successfully!


In [2]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

# ==========================
# PATHS
# ==========================
dataset_path = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\smart_city_dataset"
output_path = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\final_dataset"

# Create folders
for split in ['train', 'val', 'test']:
    for category in ['recyclable', 'non_recyclable']:
        os.makedirs(
            os.path.join(output_path, split, category),
            exist_ok=True
        )

# ==========================
# SPLIT DATA
# ==========================
categories = ['recyclable', 'non_recyclable']

for category in categories:
    category_path = os.path.join(dataset_path, category)

    images = [
        img for img in os.listdir(category_path)
        if img.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    random.shuffle(images)

    # 70% train, 15% val, 15% test
    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=0.50,
        random_state=42
    )

    split_dict = {
        'train': train_imgs,
        'val': val_imgs,
        'test': test_imgs
    }

    # Copy images
    for split_name, split_images in split_dict.items():
        split_folder = os.path.join(
            output_path,
            split_name,
            category
        )

        for img in split_images:
            src = os.path.join(category_path, img)
            dst = os.path.join(split_folder, img)
            shutil.copy(src, dst)

print("Train/Validation/Test split completed!")

Train/Validation/Test split completed!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ==========================
# PATHS
# ==========================
train_dir = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\final_dataset\\train"
val_dir = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\final_dataset\\val"
test_dir = "C:\\Users\\user\\Documents\\ECO - FRIENDLY CAPSTONE PROJECT\\DATASET\\final_dataset\\test"

# ==========================
# IMAGE SETTINGS
# ==========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ==========================
# DATA AUGMENTATION
# ==========================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)


val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# ==========================
# MODEL
# ==========================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# ==========================
# COMPILE
# ==========================
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==========================
# CALLBACKS
# ==========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# ==========================
# TRAIN MODEL
# ==========================
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=[early_stop, checkpoint]
)

# ==========================
# TEST MODEL
# ==========================
loss, accuracy = model.evaluate(test_data)

print(f"Test Accuracy: {accuracy*100:.2f}%")

Found 31625 images belonging to 2 classes.
Found 15384 images belonging to 2 classes.
Found 14340 images belonging to 2 classes.
Epoch 1/25
989/989 ━━━━━━━━━━━━━━━━━━━━ 0s 631ms/step - accuracy: 0.8567 - loss: 0.3283
Epoch 1: val_accuracy improved from None to 0.93357, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
989/989 ━━━━━━━━━━━━━━━━━━━━ 860s 868ms/step - accuracy: 0.9004 - loss: 0.2483 - val_accuracy: 0.9336 - val_loss: 0.1794
Epoch 2/25
989/989 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9292 - loss: 0.1853
Epoch 2: val_accuracy improved from 0.93357 to 0.94306, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
989/989 ━━━━━━━━━━━━━━━━━━━━ 527s 528ms/step - accuracy: 0.9308 - loss: 0.1801 - val_accuracy: 0.9431 - val_loss: 0.1517
Epoch 3/25
989/989 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - accuracy: 0.9378 - loss: 0.1635
Epoch 3: val_accuracy improved from 0.94306 to 0.95040, saving model to best_model.ker